In [5]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import matplotlib.pyplot as plt
import xgboost as xgb

from functools import partial

import sys
sys.path.append('../src')

from load_clean_data import load_clean_data
from backtest import get_cutoffs, split_data_by_cutoff, run_backtest
from baselines import add_all_baselines
from evaluation import score_forecasts, merge_backtests
from features import build_features, FEATURE_COLUMNS
from models import add_xgb_forecast, XGB_PARAMS, add_lgbm_forecast, LGBM_PARAMS

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [2]:
df = load_clean_data()
df_with_features = build_features(df)
df_with_features['StoreID'].dtype

CategoricalDtype(categories=[   1,    2,    3,    4,    5,    6,    7,    8,    9,   10,
                  ...
                  1106, 1107, 1108, 1109, 1110, 1111, 1112, 1113, 1114, 1115],
, ordered=False, categories_dtype=int64)

In [7]:
cutoffs = get_cutoffs(df['Date'].max(), 7)
first_cutoff = cutoffs[:1]

In [ ]:
add_xgb_storeid = partial(add_xgb_forecast, feature_cols = FEATURE_COLUMNS + ['StoreID'])

bt = run_backtest(df_with_features, first_cutoff, 7, add_xgb_storeid)

xgb_storeid_bt = bt.rename(columns={'xgb_forecast': 'xgb_storeid_forecast'})

xgb_storeid_bt = xgb_storeid_bt[['Store', 'Date', 'Sales', 'cutoff', 'day_ahead', 'xgb_storeid_forecast']]

bt = run_backtest(df_with_features, first_cutoff, 7, add_xgb_forecast)

print(score_forecasts(bt, ['xgb_forecast']))

print(score_forecasts(xgb_storeid_bt, ['xgb_storeid_forecast']))

          Model         RMSE         MAE       WAPE
0  xgb_forecast  1053.181152  688.492249  10.630744
                  Model        RMSE         MAE      WAPE
0  xgb_storeid_forecast  926.036316  590.973083  9.124988


In [11]:
FORECAST = True

if FORECAST:
    add_xgb_storeid = partial(add_xgb_forecast, feature_cols = FEATURE_COLUMNS + ['StoreID'])

    bt = run_backtest(df_with_features, cutoffs, 7, add_xgb_storeid)

    xgb_storeid_bt = bt.rename(columns={'xgb_forecast': 'xgb_storeid_forecast'})

    xgb_storeid_bt = xgb_storeid_bt[['Store', 'Date', 'Sales', 'cutoff', 'day_ahead', 'xgb_storeid_forecast']]

    xgb_storeid_bt.to_csv('../results/backtests/xgb_storeid_backtest.csv', index=False)

else: 
    xgb_storeid_bt = pd.read_csv('../results/backtests/xgb_storeid_backtest.csv', parse_dates=['Date', 'cutoff'], low_memory=False)

In [12]:
SAVE_SCORES = True

if SAVE_SCORES:
    scores = score_forecasts(xgb_storeid_bt, ['xgb_storeid_forecast'])
    scores.to_csv('../results/xgb_storeid_scores.csv', index=False)

In [14]:
add_lgbm_storeid = partial(add_lgbm_forecast, feature_cols = FEATURE_COLUMNS + ['StoreID'])

bt = run_backtest(df_with_features, first_cutoff, 7, add_lgbm_storeid)

lgbm_storeid_bt = bt.rename(columns={'lgbm_forecast': 'lgbm_storeid_forecast'})

lgbm_storeid_bt = lgbm_storeid_bt[['Store', 'Date', 'Sales', 'cutoff', 'day_ahead', 'lgbm_storeid_forecast']]

bt = run_backtest(df_with_features, first_cutoff, 7, add_lgbm_forecast)

print(score_forecasts(bt, ['lgbm_forecast']))

print(score_forecasts(lgbm_storeid_bt, ['lgbm_storeid_forecast']))

           Model         RMSE         MAE       WAPE
0  lgbm_forecast  1167.040164  764.012132  11.796817
                   Model         RMSE         MAE       WAPE
0  lgbm_storeid_forecast  1092.420699  714.826779  11.037365


In [15]:
FORECAST = True

if FORECAST:
    add_lgbm_storeid = partial(add_lgbm_forecast, feature_cols = FEATURE_COLUMNS + ['StoreID'])

    bt = run_backtest(df_with_features, cutoffs, 7, add_lgbm_storeid)

    lgbm_storeid_bt = bt.rename(columns={'lgbm_forecast': 'lgbm_storeid_forecast'})

    lgbm_storeid_bt = lgbm_storeid_bt[['Store', 'Date', 'Sales', 'cutoff', 'day_ahead', 'lgbm_storeid_forecast']]

    lgbm_storeid_bt.to_csv('../results/backtests/lgbm_storeid_backtest.csv', index=False)

else: 
    lgbm_storeid_bt = pd.read_csv('../results/backtests/lgbm_storeid_backtest.csv', parse_dates=['Date', 'cutoff'], low_memory=False)

In [16]:
SAVE_SCORES = True

if SAVE_SCORES:
    scores = score_forecasts(lgbm_storeid_bt, ['lgbm_storeid_forecast'])
    scores.to_csv('../results/lgbm_storeid_scores.csv', index=False)

In [17]:
combined_bt = merge_backtests(['baselines', 'xgb', 'lgbm', 'xgb_tuned', 'lgbm_tuned', 'xgb_storeid', 'lgbm_storeid'])

forecasts = ['naive_baseline', 'avg_28d_baseline', 'avg_all_baseline', 'xgb_forecast', 'xgb_tuned_forecast', 'xgb_storeid_forecast', 'lgbm_forecast', 'lgbm_tuned_forecast', 'lgbm_storeid_forecast']

combined_scores = score_forecasts(combined_bt, forecasts)

model_names = {
    'naive_baseline': 'Naive Baseline',
    'avg_28d_baseline': '28-Day Average Baseline',
    'avg_all_baseline': 'Full History Average Baseline',
    'xgb_forecast': 'XGBoost Forecast',
    'xgb_tuned_forecast': 'XGBoost Tuned',
    'xgb_storeid_forecast': 'XGBoost with StoreID',
    'lgbm_forecast': 'LightGBM Forecast',
    'lgbm_tuned_forecast': 'LightGBM Tuned',
    'lgbm_storeid_forecast': 'LightGBM with StoreID'
}

relabelled_scores = combined_scores.copy()

relabelled_scores['Model'] = relabelled_scores['Model'].replace(model_names)

relabelled_scores.sort_values(by='WAPE').style.hide(axis='index').format({ 'WAPE': '{:.2f}%' }, precision=2)

Model,RMSE,MAE,WAPE
XGBoost with StoreID,922.86,555.70,9.48%
LightGBM with StoreID,961.10,580.75,9.91%
LightGBM Tuned,994.37,597.28,10.19%
XGBoost Forecast,1001.30,604.95,10.33%
XGBoost Tuned,992.88,605.63,10.34%
LightGBM Forecast,1035.76,619.47,10.57%
Full History Average Baseline,2029.48,1261.49,21.53%
28-Day Average Baseline,2127.55,1326.10,22.63%
Naive Baseline,3206.68,2112.59,36.06%


In [18]:
first_10 = cutoffs[:10]
last_15_bt = combined_bt[~combined_bt['cutoff'].isin(first_10)]

assert last_15_bt['cutoff'].nunique() == 15, 'not 15 cutoffs'
assert len(last_15_bt) == 117075, 'wrong length'

last_15_scores = score_forecasts(last_15_bt, forecasts)

relabelled_scores = last_15_scores.copy()

relabelled_scores['Model'] = relabelled_scores['Model'].replace(model_names)

relabelled_scores.sort_values(by='WAPE').style.hide(axis='index').format({ 'WAPE': '{:.2f}%' }, precision=2)

Model,RMSE,MAE,WAPE
XGBoost with StoreID,870.71,551.70,9.33%
LightGBM with StoreID,915.32,575.86,9.74%
LightGBM Tuned,949.23,593.35,10.03%
XGBoost Forecast,956.45,599.60,10.14%
XGBoost Tuned,961.86,608.98,10.30%
LightGBM Forecast,1011.97,620.17,10.49%
Full History Average Baseline,2019.27,1278.61,21.62%
28-Day Average Baseline,2151.14,1368.45,23.14%
Naive Baseline,3312.36,2170.45,36.70%


In [ ]:
pairs = {
    'xgboost': ('xgb_forecast', 'xgb_storeid_forecast'),
    'lightgbm': ('lgbm_forecast', 'lgbm_storeid_forecast'),
}

cols = [c for pair in pairs.values() for c in pair]

by_cutoff = score_forecasts(combined_bt, cols, by='cutoff')

pivotted = by_cutoff.pivot(index='cutoff', columns='Model', values='WAPE')

for name, (default_col, storeid_col) in pairs.items():
    gain = pivotted[default_col] - pivotted[storeid_col]   # positive = store id is better

    print(name, ': store id better in', (gain > 0).sum(), 'of', len(gain), 'weeks')
    print('average gain per week:', round(gain.mean(), 2))
    print('best weeks:')
    print(gain.sort_values(ascending=False).head(3))
    print('worst weeks:')
    print(gain.sort_values().head(3))
    print()